# Preprocesamiento de datos

**Curso:** Machine learning — aprendizaje supervisado de regresión en Python
**Sesión 2** · Andrés Felipe Puerta Vélez

---

Este cuaderno acompaña las diapositivas de la sesión. Recorre, paso a paso, lo
que hay que hacerle a unos datos antes de que un modelo pueda aprender de ellos:

1. Primer contacto con los datos
2. Valores faltantes: detectar, entender, eliminar o imputar
3. Partir en entrenamiento y prueba (y por qué va antes que casi todo)
4. Imputación en serio
5. Variables categóricas: label, ordinal y one-hot encoding
6. Agrupación y discretización
7. Escalado
8. Transformar variables sesgadas
9. Todo junto: `Pipeline`, `ColumnTransformer` y una comparación final

**Cómo ejecutarlo.** Si descargaron la carpeta `data/` junto al cuaderno, corre
sin conexión. Si no, se descarga solo de la web del curso.

## 0 · Preparación

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

print("pandas", pd.__version__)
print("numpy ", np.__version__)

pandas 2.3.3
numpy  2.2.6


In [2]:
URL = "https://afpuertav.github.io/courses/machine_learning_regresion/Preprocesamiento/data/"

def cargar(nombre):
    """Lee de la carpeta data/ si está; si no, de la web del curso."""
    try:
        return pd.read_csv("data/" + nombre)
    except FileNotFoundError:
        return pd.read_csv(URL + nombre)

cars = cargar("cars93.csv")
dia  = cargar("diamonds.csv.gz")

print("Cars93  ", cars.shape)
print("diamonds", dia.shape)

Cars93   (93, 27)
diamonds (53940, 10)


### Los dos conjuntos

| | Cars93 | diamonds |
|---|---|---|
| Filas × columnas | 93 × 27 | 53.940 × 10 |
| Objetivo | `Price` (miles de USD) | `price` (USD) |
| Aporta | vacíos reales, categóricas nominales, alta cardinalidad | categóricas **ordinales** reales, volumen |

---
## 1 · Primer contacto

Antes de transformar nada: mirar. Cuántas filas, qué columnas, de qué tipo, qué
falta y qué valores raros hay. Casi todos los errores caros del preprocesamiento
son errores de no haber mirado, y ninguno de ellos da un mensaje de error.

In [3]:
cars.head()

,Manufacturer,Model,Type,Min.Price,Price,Max.Price,MPG.city,MPG.highway,AirBags,DriveTrain,Cylinders,EngineSize,Horsepower,RPM,Rev.per.mile,Man.trans.avail,Fuel.tank.capacity,Passengers,Length,Wheelbase,Width,Turn.circle,Rear.seat.room,Luggage.room,Weight,Origin,Make
0,Acura,Integra,Small,12.9,15.9,18.8,25,31,NaN,Front,4,1.8,140,6300,2890,Yes,13.2,5,177,102,68,37,26.5,11.0,2705,non-USA,Acura Integra
1,Acura,Legend,Midsize,29.2,33.9,38.7,18,25,Driver & Passenger,Front,6,3.2,200,5500,2335,Yes,18.0,5,195,115,71,38,30.0,15.0,3560,non-USA,Acura Legend
2,Audi,90,Compact,25.9,29.1,32.3,20,26,Driver only,Front,6,2.8,172,5500,2280,Yes,16.9,5,180,102,67,37,28.0,14.0,3375,non-USA,Audi 90
3,Audi,100,Midsize,30.8,37.7,44.6,19,26,Driver & Passenger,Front,6,2.8,172,5500,2535,Yes,21.1,6,193,106,70,37,31.0,17.0,3405,non-USA,Audi 100
4,BMW,535i,Midsize,23.7,30.0,36.2,22,30,Driver only,Rear,4,3.5,208,5700,2545,Yes,21.1,4,186,109,69,39,27.0,13.0,3640,non-USA,BMW 535i


In [4]:
cars[["Manufacturer", "Type", "Price", "AirBags",
      "Cylinders", "Horsepower", "Luggage.room", "Origin"]].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 93 entries, 0 to 92
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Manufacturer  93 non-null     object 
 1   Type          93 non-null     object 
 2   Price         93 non-null     float64
 3   AirBags       59 non-null     object 
 4   Cylinders     93 non-null     object 
 5   Horsepower    93 non-null     int64  
 6   Luggage.room  82 non-null     float64
 7   Origin        93 non-null     object 
dtypes: float64(2), int64(1), object(5)
memory usage: 5.9+ KB


Dos cosas saltan a la vista:

- **`AirBags` solo tiene 59 valores de 93.** Falta más de un tercio.
- **`Cylinders` es `object`**, o sea texto, cuando uno esperaría un número.
  Ya veremos por qué.

In [5]:
cars[["Price", "MPG.city", "Horsepower", "Weight"]].describe().round(2)

,Price,MPG.city,Horsepower,Weight
count,93.00,93.00,93.00,93.0
mean,19.51,22.37,143.83,3072.9
std,9.66,5.62,52.37,589.9
min,7.40,15.00,55.00,1695.0
25%,12.20,18.00,103.00,2620.0
50%,17.70,21.00,140.00,3040.0
75%,23.30,25.00,170.00,3525.0
max,61.90,46.00,300.00,4105.0


`Weight` vive en los miles y `MPG.city` en las decenas. Guarden esa idea: es
exactamente el problema que resolverá el **escalado** en la sección 7.

---
## Chequeo del capítulo 1 · El banco de pruebas

Al cerrar **cada capítulo** vamos a medir lo mismo, de la misma forma:

- **Dos modelos.** `LinearRegression()` y una red `MLPRegressor` de una capa de
  32 neuronas. Siempre los mismos, sin tocarles nada entre capítulos.
- **Dos métricas.** *MSE*, el error cuadrático medio en (miles de USD)², y
  *R²*, cuánta variación del precio explica el modelo: 1 es perfecto, 0 es tan
  bueno como decir siempre la media, y negativo es peor que eso.
- **Validación cruzada de 5 pliegues** sobre los 93 coches. Con una sola
  partición de 19 coches las cifras bailan tanto que no se podría comparar nada.

In [6]:
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import cross_val_score

# Las herramientas que iremos usando capítulo a capítulo. Cada una se explica
# a fondo cuando le toca; aquí solo se importan para que el banco funcione.
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

def modelos():
    """Los dos modelos del banco. Idénticos en los ocho capítulos."""
    return [("LinearRegression", LinearRegression()),
            ("MLPRegressor", MLPRegressor(hidden_layer_sizes=(32,),
                                          max_iter=2000,
                                          random_state=7))]

historial = []

def chequeo(capitulo, tuberia_de, X, y):
    """Mide los dos modelos y acumula el resultado en el historial."""
    fila = [capitulo]
    for nombre, modelo in modelos():
        try:
            t = tuberia_de(modelo)
            # error_score="raise" deja pasar el error de verdad; si no,
            # cross_val_score lo tapa con un "All the 5 fits failed"
            mse = -cross_val_score(t, X, y, cv=5, error_score="raise",
                                   scoring="neg_mean_squared_error").mean()
            r2  =  cross_val_score(t, X, y, cv=5, error_score="raise",
                                   scoring="r2").mean()
            fila += [round(mse, 2), round(r2, 3)]
        except Exception as e:
            fila += ["error", "error"]
            motivo = str(e).strip().splitlines()[0]
            print(f"  {nombre}: {type(e).__name__}: {motivo[:60]}")
    historial.append(fila)
    return pd.DataFrame(historial, columns=["capítulo", "MSE lineal",
                                            "R2 lineal", "MSE red", "R2 red"])

### Chequeo 1 · Sin preprocesar nada

Probamos los datos tal cual salen del CSV.

In [7]:
# Quitamos las columnas que son el precio disfrazado o identificadores.
# (En el capítulo 3 volveremos sobre esto al hacer la partición.)
cars_m = cars.drop(columns=["Min.Price", "Max.Price", "Make", "Model"])

X = cars_m.drop(columns=["Price"])
y = cars_m["Price"]

num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(exclude=np.number).columns.tolist()
print(len(num_cols), "numéricas y", len(cat_cols), "categóricas")

chequeo("1a · Sin tocar nada", lambda m: m, X, y)

15 numéricas y 7 categóricas
  LinearRegression: ValueError: could not convert string to float: 'Chrylser'
  MLPRegressor: ValueError: could not convert string to float: 'Chrylser'


,capítulo,MSE lineal,R2 lineal,MSE red,R2 red
0,1a · Sin tocar nada,error,error,error,error


In [8]:
chequeo("1b · Solo las numéricas", lambda m: m, X[num_cols], y)

  LinearRegression: ValueError: Input X contains NaN.
  MLPRegressor: ValueError: Input X contains NaN.


,capítulo,MSE lineal,R2 lineal,MSE red,R2 red
0,1a · Sin tocar nada,error,error,error,error
1,1b · Solo las numéricas,error,error,error,error


**No es que el modelo prediga mal: es que no arranca.** Y los dos errores nos
marcan el orden del día:

- `could not convert string to float: 'Nissan'` — hay texto donde el modelo
  espera números. Lo resolveremos en el capítulo 5.
- `Input X contains NaN` — hay huecos. Es el capítulo que empieza ahora.

---
## 2 · Valores faltantes

### 2.1 Por qué hay que ocuparse de esto

La razón inmediata es que scikit-learn **no acepta vacíos**: la mayoría de sus
modelos fallan en cuanto encuentran un `NaN`.

In [9]:
from sklearn.linear_model import LinearRegression

try:
    LinearRegression().fit(cars[["Horsepower", "Luggage.room"]], cars["Price"])
except ValueError as e:
    print("ValueError:", str(e).split("\n")[0])

ValueError: Input X contains NaN.


La razón de fondo es más interesante: **cómo rellenamos esos huecos cambia lo que
el modelo aprende**. No es un trámite para que deje de dar error.

### 2.2 Detectarlos

In [10]:
faltan = cars.isna().sum()
print(faltan[faltan > 0], "\n")

pct = (cars.isna().mean() * 100).round(1)
print(pct[pct > 0].sort_values(ascending=False))

AirBags           34
Rear.seat.room     2
Luggage.room      11
dtype: int64 

AirBags           36.6
Luggage.room      11.8
Rear.seat.room     2.2
dtype: float64


El porcentaje importa más que el conteo: 34 vacíos sobre 93 filas es más de un
tercio de la columna.

### 2.3 No todos los vacíos son iguales

Antes de rellenar hay que preguntarse **por qué** falta el dato. La estadística
distingue tres mecanismos:

| Sigla | Significa | Qué quiere decir | Ejemplo |
|---|---|---|---|
| **MCAR** | *Missing Completely At Random* | Falta completamente al azar: la ausencia no depende de nada | Se perdió una hoja del cuestionario |
| **MAR** | *Missing At Random* | Falta al azar **condicionado a lo observado**: depende de otras columnas que sí tenemos | A las furgonetas nunca les midieron el maletero |
| **MNAR** | *Missing Not At Random* | **No** falta al azar: depende del propio valor que falta | Los de renta alta no declaran su renta |

Y lo que implica cada uno:

- **MCAR** → eliminar filas es válido; imputar con la media no sesga.
- **MAR** → eliminar **sí** sesga. Hay que imputar usando las columnas
  relacionadas: por grupo, o con KNN.
- **MNAR** → no hay solución estadística limpia. Lo honesto es añadir una
  columna que marque que faltaba, y reconocer la limitación.

> ⚠️ Imputar con la media asume MCAR. Si el mecanismo es MNAR, imputar
> **introduce sesgo** en vez de arreglarlo.

### 2.4 El caso de `AirBags`

In [11]:
cars["AirBags"].value_counts(dropna=False)

AirBags
Driver only           43
NaN                   34
Driver & Passenger    16
Name: count, dtype: int64

Aquí el `NaN` **no es un dato perdido**: es el coche que no traía airbag. En el
dataset original esa categoría era `"None"` y al leerla se convirtió en vacío.

Imputar esos 34 con la moda sería inventarse 34 airbags que no existen. Lo
correcto es tratarlos como una categoría propia.

**La pregunta correcta no es "¿con qué lo relleno?" sino "¿por qué no está?".**

| Si el vacío significa… | Entonces… |
|---|---|
| No se midió | Imputar tiene sentido |
| No aplica | Es una categoría, no un vacío |
| Se perdió | Imputar, dejando constancia |
| Es un cero mal codificado | Se corrige, no se imputa |

### 2.5 Opción 1 · Eliminar

In [12]:
print("original         ", cars.shape)
print("dropna() filas   ", cars.dropna().shape)
print("dropna(axis=1)   ", cars.dropna(axis=1).shape)
print("thresh 90% cols  ", cars.dropna(axis=1, thresh=int(0.9 * len(cars))).shape)

original          (93, 27)
dropna() filas    (54, 27)
dropna(axis=1)    (93, 24)
thresh 90% cols   (93, 25)


`dropna()` nos deja **54 de 93 filas**: perdemos el 42 % de los datos por culpa
de tres columnas. Casi nunca es la opción correcta.

**¿Cuándo sí eliminar?**

- **La columna**: cuando falta muchísimo (>50-60 %) y no es clave.
- **La fila**: cuando son poquísimas (<2-5 %) y el faltante es MCAR.
- **La fila, siempre**: si lo que falta es la variable objetivo. No se imputa lo
  que hay que predecir.
- **No eliminar**: cuando el hecho de faltar es informativo.

### 2.6 Opción 2 · Imputar: ¿media, mediana o moda?

In [13]:
lr = cars["Luggage.room"]
print(f"media    {lr.mean():.3f}")
print(f"mediana  {lr.median():.3f}")
print(f"moda     {lr.mode()[0]:.3f}")

media    13.890
mediana  14.000
moda     14.000


Aquí dan casi lo mismo porque la variable es simétrica. En una variable sesgada
(ingresos, precios) la **media se va detrás de los atípicos** y la mediana es la
opción segura.

**Regla práctica:** numérica simétrica → media; numérica sesgada → mediana;
categórica → moda o una categoría nueva.

---
## Chequeo del capítulo 2 · Eliminando los vacíos

Nos quedamos con las numéricas y tiramos las filas incompletas.

In [14]:
X_limpio = X[num_cols].dropna()
y_limpio = y.loc[X_limpio.index]
print("filas que sobreviven:", len(X_limpio), "de", len(X))

chequeo("2 · dropna sobre las numéricas", lambda m: m, X_limpio, y_limpio)

filas que sobreviven: 82 de 93


,capítulo,MSE lineal,R2 lineal,MSE red,R2 red
0,1a · Sin tocar nada,error,error,error,error
1,1b · Solo las numéricas,error,error,error,error
2,2 · dropna sobre las numéricas,46.77,0.295,68165.39,-1015.81


Ya tenemos números, y no son buenos:

- **La lineal arranca.** $R^2 = 0{,}295$: explica menos de un tercio de la
  variación del precio, pero al menos funciona.
- **La red es un desastre.** $R^2 = -1015$. Un negativo significa peor que
  predecir siempre la media; mil veces peor, en este caso.
- **Y hemos perdido 11 coches**, un 12 % de los datos, por culpa de dos columnas
  incompletas.

Los tres problemas tienen arreglo. Vamos con ellos por orden.

---
## 3 · Partir antes de transformar

### La fuga de datos

Hay **fuga de datos** (*data leakage*) cuando en el entrenamiento se cuela
información que en el momento de predecir no estaría disponible. El resultado es
un modelo que parece buenísimo en pruebas y fracasa en producción.

Imputar con la media de **todo** el dataset es fuga: esa media contiene
información de las filas de prueba, que se supone que aún no hemos visto.

In [15]:
# X e y ya vienen del banco de pruebas, sin las columnas que eran
# el precio disfrazado ni los identificadores
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print("X_train", X_train.shape, " X_test", X_test.shape)
print("y_train", y_train.shape, "  y_test", y_test.shape)

X_train (74, 22)  X_test (19, 22)
y_train (74,)   y_test (19,)


In [16]:
print(f"media de Luggage.room en TODO el dataset : {cars_m['Luggage.room'].mean():.3f}")
print(f"media de Luggage.room solo en train      : {X_train['Luggage.room'].mean():.3f}")

media de Luggage.room en TODO el dataset : 13.890
media de Luggage.room solo en train      : 14.212


Son valores distintos. Usar el primero para rellenar el conjunto de prueba
significa que la prueba ya sabe algo de sí misma.

### La regla, en una frase

> **Todo lo que aprenda un parámetro de los datos** —medias, medianas, mínimos,
> máximos, categorías, desviaciones— se ajusta **solo con entrenamiento** y se
> aplica a prueba.

- `fit` en train → aprende los parámetros.
- `transform` en train y en test → aplica lo aprendido.
- **Nunca** `fit` ni `fit_transform` en test.

### El orden correcto

1. Cargar los datos y mirarlos.
2. Arreglar lo que es objetivamente un error: tipos, duplicados, unidades.
3. **Partir en entrenamiento y prueba.**
4. `fit` de imputadores, codificadores y escaladores con entrenamiento.
5. `transform` de ambos conjuntos.
6. Entrenar y evaluar.

El paso 2 puede ir antes de partir porque **no aprende nada de los datos**.

In [17]:
num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=np.number).columns.tolist()

print("numéricas  :", num_cols)
print()
print("categóricas:", cat_cols)

numéricas  : ['MPG.city', 'MPG.highway', 'EngineSize', 'Horsepower', 'RPM', 'Rev.per.mile', 'Fuel.tank.capacity', 'Passengers', 'Length', 'Wheelbase', 'Width', 'Turn.circle', 'Rear.seat.room', 'Luggage.room', 'Weight']

categóricas: ['Manufacturer', 'Type', 'AirBags', 'DriveTrain', 'Cylinders', 'Man.trans.avail', 'Origin']


---
## Chequeo del capítulo 3 · Con fuga y sin fuga

Rellenamos con la mediana de dos maneras: calculándola con **todo** el dataset
(fuga, porque esa mediana ha visto los cinco pliegues) y calculándola **dentro**
de la tubería, donde se recalcula en cada pliegue.

In [18]:
X_fuga = X[num_cols].fillna(X[num_cols].median())
chequeo("3a · Imputando fuera (con fuga)", lambda m: m, X_fuga, y)

,capítulo,MSE lineal,R2 lineal,MSE red,R2 red
0,1a · Sin tocar nada,error,error,error,error
1,1b · Solo las numéricas,error,error,error,error
2,2 · dropna sobre las numéricas,46.77,0.295,68165.39,-1015.81
3,3a · Imputando fuera (con fuga),37.39,0.44,71652.15,-1132.738


In [19]:
chequeo("3b · Imputando dentro (correcto)",
        lambda m: Pipeline([("imp", SimpleImputer(strategy="median")),
                            ("reg", m)]),
        X[num_cols], y)

,capítulo,MSE lineal,R2 lineal,MSE red,R2 red
0,1a · Sin tocar nada,error,error,error,error
1,1b · Solo las numéricas,error,error,error,error
2,2 · dropna sobre las numéricas,46.77,0.295,68165.39,-1015.81
3,3a · Imputando fuera (con fuga),37.39,0.44,71652.15,-1132.738
4,3b · Imputando dentro (correcto),37.39,0.44,71652.37,-1132.739


**Salen iguales. Y eso es justo lo peligroso.**

La fuga no se nota en las métricas. Si se notara, la cazaríamos enseguida: el
problema es que se cuela sin dejar rastro.

Aquí no cambia nada porque la mediana es muy robusta — quitar 20 coches apenas
la mueve. Pero el mismo error con otras técnicas sí hace daño:

- **Target encoding:** la media del objetivo por categoría se contamina de lleno
  con las filas de prueba.
- **Selección de variables:** elegir columnas mirando todo el dataset ya es
  haber usado el conjunto de prueba.
- **Escalado con atípicos:** un solo valor extremo en prueba mueve el mínimo y
  el máximo.

La regla no se sigue porque se vea el daño, sino **porque no se ve**.

---
## 4 · Imputar en serio

### 4.1 `SimpleImputer`

Hacerlo con pandas funciona, pero deja los valores de relleno sueltos en una
variable. `SimpleImputer` los **guarda dentro del objeto**, y eso es lo que
permite aplicar exactamente lo mismo al conjunto de prueba y a producción.

In [20]:
from sklearn.impute import SimpleImputer

# estrategias: "mean", "median", "most_frequent", "constant"
imp_num = SimpleImputer(strategy="median")
imp_num.fit(X_train[num_cols])          # aprende SOLO con entrenamiento

pd.Series(imp_num.statistics_, index=num_cols)[["Rear.seat.room", "Luggage.room"]]

Rear.seat.room    28.0
Luggage.room      14.0
dtype: float64

In [21]:
Xtr_num = pd.DataFrame(imp_num.transform(X_train[num_cols]),
                       columns=num_cols, index=X_train.index)
Xte_num = pd.DataFrame(imp_num.transform(X_test[num_cols]),   # transform, NO fit
                       columns=num_cols, index=X_test.index)

print("NAs en train antes :", int(X_train[num_cols].isna().sum().sum()))
print("NAs en train después:", int(Xtr_num.isna().sum().sum()))
print("NAs en test después :", int(Xte_num.isna().sum().sum()))

NAs en train antes : 9
NAs en train después: 0
NAs en test después : 0


### 4.2 Imputar categóricas

In [22]:
imp_cat = SimpleImputer(strategy="constant", fill_value="Sin airbag")
Xtr_cat = pd.DataFrame(imp_cat.fit_transform(X_train[cat_cols]),
                       columns=cat_cols, index=X_train.index)

Xtr_cat["AirBags"].value_counts()

AirBags
Driver only           35
Sin airbag            26
Driver & Passenger    13
Name: count, dtype: int64

Con `strategy="constant"` creamos una categoría explícita en lugar de disfrazar
los vacíos de moda. Es justo lo que pedía el caso de `AirBags`.

### 4.3 `KNNImputer`

En vez de un único valor para todos, busca los **k coches más parecidos** en las
demás columnas y promedia su valor. Sirve cuando el faltante es MAR: aprovecha
precisamente esas columnas de las que depende la ausencia.

In [23]:
from sklearn.impute import KNNImputer

knn = KNNImputer(n_neighbors=5)
Xtr_knn = pd.DataFrame(knn.fit_transform(X_train[num_cols]),
                       columns=num_cols, index=X_train.index)

faltaban = X_train.index[X_train["Luggage.room"].isna()][:4]
pd.DataFrame({
    "mediana": Xtr_num.loc[faltaban, "Luggage.room"],
    "knn":     Xtr_knn.loc[faltaban, "Luggage.room"].round(2),
})

,mediana,knn
65,14.0,17.6
15,14.0,16.6
69,14.0,16.6
16,14.0,19.0


La mediana pone 14 a todos. El KNN nota que esos cuatro son coches grandes y les
pone entre 16 y 19. Más fiel, pero más lento y **exige escalar antes**, porque el
vecindario se mide con distancias.

### 4.4 Imputar por grupo, y su trampa

In [24]:
cars_m.groupby("Type")["Luggage.room"].median()

Type
Compact    14.0
Large      18.0
Midsize    15.0
Small      12.0
Sporty     11.5
Van         NaN
Name: Luggage.room, dtype: float64

`Van` sale `NaN`: **a las nueve furgonetas les falta el dato a todas**. Si el
grupo entero está vacío, la imputación por grupo no rellena nada y hay que caer
en un valor global.

Y de paso confirma que el faltante era **MAR**: depende de `Type`, que sí
observamos.

### 4.5 El catálogo completo

- **`IterativeImputer`** — modela cada columna con vacíos a partir de las demás
  e itera hasta converger. Lo más potente y lo más caro.
- **Indicador de faltante** — `SimpleImputer(add_indicator=True)` añade una
  columna 0/1; deja que el modelo aprenda si faltar significa algo.
- **No imputar** — `HistGradientBoostingRegressor`, LightGBM y XGBoost manejan
  `NaN` de forma nativa.

---
## Chequeo del capítulo 4 · Con imputación

In [25]:
chequeo("4 · SimpleImputer(mediana)",
        lambda m: Pipeline([("imp", SimpleImputer(strategy="median")),
                            ("reg", m)]),
        X[num_cols], y)

,capítulo,MSE lineal,R2 lineal,MSE red,R2 red
0,1a · Sin tocar nada,error,error,error,error
1,1b · Solo las numéricas,error,error,error,error
2,2 · dropna sobre las numéricas,46.77,0.295,68165.39,-1015.81
3,3a · Imputando fuera (con fuga),37.39,0.44,71652.15,-1132.738
4,3b · Imputando dentro (correcto),37.39,0.44,71652.37,-1132.739
5,4 · SimpleImputer(mediana),37.39,0.44,71652.37,-1132.739


La lineal mejora de verdad: el MSE baja de 46,77 a 37,39 y el $R^2$ sube de
0,295 a 0,440. No porque imputar sea mágico, sino porque **entrenamos con 93
coches en vez de 82**.

La red sigue igual de rota. Su problema es otro y todavía no lo hemos tocado.

---
## 5 · Variables categóricas

### 5.1 Por qué hay que codificar

In [26]:
try:
    LinearRegression().fit(X_train, y_train)
except ValueError as e:
    print("ValueError:", str(e).split("\n")[0])

ValueError: could not convert string to float: 'Nissan'


Un modelo de regresión multiplica cada columna por un coeficiente y suma. La
palabra `"Nissan"` no se puede multiplicar por nada. **Codificar** es convertir
esas etiquetas en números **sin inventarse relaciones que no existen**. Esa
última condición es toda la dificultad del asunto.

### 5.2 Tres tipos de categórica

| Tipo | Qué es | Ejemplos | Codificación |
|---|---|---|---|
| **Nominal** | Sin orden | `Type`, `Manufacturer`, `DriveTrain` | one-hot |
| **Ordinal** | Con orden real | pureza de un diamante, talla S/M/L | ordinal con orden declarado |
| **Binaria** | Dos valores | `Origin`, `Man.trans.avail` | una columna 0/1 |

La decisión no la toma el código: la toma quien conoce el dominio.

In [27]:
X_train[cat_cols].nunique().sort_values(ascending=False)

Manufacturer       31
Type                6
Cylinders           6
DriveTrain          3
AirBags             2
Man.trans.avail     2
Origin              2
dtype: int64

`Manufacturer` tiene **31 niveles distintos en 74 filas de entrenamiento**.
Guarden ese dato: más abajo va a hacer explotar el modelo.

### 5.3 Binaria

In [28]:
(X_train["Origin"] == "USA").astype(int).value_counts()

Origin
1    39
0    35
Name: count, dtype: int64

Con dos categorías, una sola columna 0/1 basta y sobra. Crear dos sería
redundante: la segunda sería siempre `1 - primera`.

### 5.4 Label encoding: ¿para qué sirve?

**Label encoding** sustituye cada categoría por un número entero. Ni más ni
menos. `LabelEncoder` los asigna **por orden alfabético**, empezando en 0.

Su uso legítimo es **la variable objetivo de un problema de clasificación**:
cuando hay que predecir `"perro"` / `"gato"` / `"pájaro"`, el modelo necesita que
esas tres etiquetas sean 0, 1 y 2, y ahí el número es solo un identificador que
nunca se opera.

El problema empieza cuando alguien lo usa en las **variables de entrada**.

In [29]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
cilindros = cars_m["Cylinders"].astype(str)
codigos = le.fit_transform(cilindros)

# la correspondencia que aprendió
pd.DataFrame({"categoría": le.classes_,
              "número asignado": range(len(le.classes_))})

,categoría,número asignado
0,3,0
1,4,1
2,5,2
3,6,3
4,8,4
5,rotary,5


In [30]:
pd.DataFrame({"Cylinders": cilindros, "codificado": codigos}).head(6)

,Cylinders,codificado
0,4,1
1,6,3
2,6,3
3,6,3
4,4,1
5,4,1


Un coche de 4 cilindros pasa a ser un 1, y uno de 6 pasa a ser un 3.

**Por qué casi siempre está mal.** El modelo lee esos números como *cantidades*,
no como etiquetas. Con esta codificación estamos afirmando tres cosas falsas:

1. Que **`rotary` (5) es mayor que 8 cilindros (4)**. Alfabéticamente tiene
   sentido; mecánicamente no significa nada.
2. Que **las distancias son iguales**: de 3 a 4 cilindros hay "1", y de 6 a 8
   también hay "1".
3. Que **el orden correcto es el alfabético**, que es una casualidad de cómo se
   escriben las palabras.

*Excepción:* los **árboles** lo toleran bastante bien, porque parten por umbrales
y no interpretan la magnitud.

### 5.5 One-hot: la alternativa para nominales

Crea una columna por categoría, con un 1 en la que corresponde y 0 en las demás.
Así ninguna categoría queda "por encima" de otra: todas están a la misma
distancia.

In [31]:
pd.get_dummies(X_train[["Type"]], prefix="Type", dtype=int).head()

,Type_Compact,Type_Large,Type_Midsize,Type_Small,Type_Sporty,Type_Van
65,0,0,0,0,0,1
15,0,0,0,0,0,1
68,0,0,1,0,0,0
78,0,0,0,1,0,0
30,0,0,0,1,0,0


`get_dummies` está bien para explorar, pero **no para producción**: si el
conjunto de prueba trae una categoría que el entrenamiento no tenía, genera un
juego de columnas distinto y el modelo recibe una matriz que no reconoce.

In [32]:
from sklearn.preprocessing import OneHotEncoder

enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop="first")
enc.fit(X_train[["Type", "DriveTrain"]])

print(enc.get_feature_names_out())
print("forma de la salida:", enc.transform(X_train[["Type", "DriveTrain"]]).shape)

['Type_Large' 'Type_Midsize' 'Type_Small' 'Type_Sporty' 'Type_Van'
 'DriveTrain_Front' 'DriveTrain_Rear']
forma de la salida: (74, 7)


`OneHotEncoder` **recuerda** las categorías del entrenamiento y siempre devuelve
las mismas columnas, en el mismo orden.

**Los dos argumentos que importan:**

- **`handle_unknown="ignore"`** — una categoría no vista en entrenamiento sale
  como una fila de ceros, en vez de lanzar una excepción en mitad de producción.
- **`drop="first"`** — quita una columna por variable. Con las seis columnas de
  `Type`, la suma de todas es siempre 1; esa **colinealidad perfecta** hace que
  existan infinitas combinaciones de coeficientes que dan el mismo resultado, y
  la regresión no sabe cuál elegir. Al quitar `Compact`, esa pasa a ser la
  categoría de referencia y los demás coeficientes se leen como *"cuánto más caro
  que un Compact"*.

  Con árboles o con regularización (Ridge, Lasso) no hace falta quitarla.

### 5.6 Alta cardinalidad: el problema de `Manufacturer`

In [33]:
vc = X_train["Manufacturer"].value_counts()
print(vc.head(6), "\n")
print("marcas con 1 solo coche:", int((vc == 1).sum()), "de", len(vc))

Manufacturer
Ford         7
Chevrolet    6
Nissan       4
Dodge        4
Pontiac      4
Buick        4
Name: count, dtype: int64 

marcas con 1 solo coche: 12 de 31


One-hot sobre esto son **31 columnas nuevas**, 12 de ellas con un único 1. De una
columna así no se aprende nada: se memoriza esa fila.

Hay cinco salidas, de la más habitual a la más exótica.

#### Salida 1 · Agrupar las categorías raras

**Cuándo:** siempre que haya una cola larga de categorías poco frecuentes y no
exista un criterio de dominio mejor. Es la primera que hay que probar.

In [34]:
frecuentes = vc[vc >= 3].index
agrupado = X_train["Manufacturer"].where(
    X_train["Manufacturer"].isin(frecuentes), "Otras")

print(agrupado.value_counts())
print("\nde", X_train["Manufacturer"].nunique(), "niveles a", agrupado.nunique())

Manufacturer
Otras         26
Ford           7
Chevrolet      6
Nissan         4
Buick          4
Mazda          4
Dodge          4
Pontiac        4
Oldsmobile     3
Subaru         3
Hyundai        3
Volkswagen     3
Toyota         3
Name: count, dtype: int64

de 31 niveles a 13


`OneHotEncoder(min_frequency=3)` lo hace solo, dentro del `Pipeline`.

#### Salida 2 · Frequency encoding

Sustituir cada categoría por **la frecuencia con que aparece**. Una sola columna
numérica, sin importar cuántos niveles haya.

**Cuándo:** cuando la popularidad de la categoría es en sí misma informativa —un
fabricante con muchos modelos es un fabricante masivo— y hay miles de niveles.
**Riesgo:** dos marcas distintas con la misma frecuencia quedan indistinguibles.

In [35]:
frec = cars_m["Manufacturer"].value_counts(normalize=True)
pd.DataFrame({
    "Manufacturer": cars_m["Manufacturer"],
    "frecuencia":   cars_m["Manufacturer"].map(frec).round(4),
}).head(6)

,Manufacturer,frecuencia
0,Acura,0.0215
1,Acura,0.0215
2,Audi,0.0215
3,Audi,0.0215
4,BMW,0.0108
5,Buick,0.0430


#### Salida 3 · Target encoding

Sustituir cada categoría por **la media de la variable objetivo** dentro de esa
categoría. Una columna, y muy informativa: le da al modelo directamente lo que la
categoría dice sobre el precio.

**Cuándo:** competiciones y modelos tabulares con cardinalidad muy alta (códigos
postales, IDs de producto). Suele ser la codificación que más aporta.

In [36]:
media_por_marca = cars_m.groupby("Manufacturer")["Price"].mean()
pd.DataFrame({
    "Manufacturer": cars_m["Manufacturer"],
    "target_enc":   cars_m["Manufacturer"].map(media_por_marca).round(2),
}).head(6)

,Manufacturer,target_enc
0,Acura,24.90
1,Acura,24.90
2,Audi,33.40
3,Audi,33.40
4,BMW,30.00
5,Buick,21.62


In [37]:
# El peligro: categorías con una sola fila
vc_full = cars_m["Manufacturer"].value_counts()
marca_unica = vc_full[vc_full == 1].index[0]
print(f"{marca_unica}: media de la categoría = {media_por_marca[marca_unica]:.1f}")
print(f"        precio real de ese único coche = "
      f"{cars_m.loc[cars_m['Manufacturer'] == marca_unica, 'Price'].iloc[0]:.1f}")

Infiniti: media de la categoría = 47.9
        precio real de ese único coche = 47.9


Para una marca con un solo coche, la "media de la categoría" **es el precio de
ese coche**. Le estamos dando al modelo la respuesta de esa fila disfrazada de
variable de entrada: fuga de datos de manual.

Se controla con **suavizado** (mezclar la media del grupo con la media global,
según cuántas filas tenga el grupo) y calculando las medias **dentro de una
validación cruzada**. `sklearn.preprocessing.TargetEncoder` ya lo trae hecho.

#### Salida 4 · Agrupar por conocimiento del dominio

Reemplazar la categoría por una **propiedad suya** que sí tenga pocos niveles y
significado:

- `Manufacturer` (31) → `Origin` (2). Ya lo tenemos en el dataset.
- `Manufacturer` (31) → gama: generalista / prémium / lujo (3).
- `Manufacturer` (31) → grupo empresarial (~10).

**Cuándo:** siempre que se pueda. Es la única salida que **añade información** en
lugar de limitarse a comprimir la que ya había, y la que mejor resiste
categorías nuevas en producción.

#### Salida 5 · Embeddings

Aprender, durante el entrenamiento de una red neuronal, un **vector de pocas
dimensiones** para cada categoría. Categorías que se comportan parecido acaban
con vectores parecidos. Es lo mismo que se hace con las palabras en
procesamiento de lenguaje.

**Cuándo:** cardinalidades de miles o decenas de miles (usuarios, productos,
ciudades) y con una red neuronal de por medio. **Cuándo no:** en este curso. Con
31 marcas y 93 coches es artillería para matar una mosca.

### 5.7 Ordinales de verdad: `diamonds`

In [38]:
for c in ["cut", "color", "clarity"]:
    print(f"{c:8s} {dia[c].nunique()} niveles: {sorted(dia[c].unique())}")

cut      5 niveles: ['Fair', 'Good', 'Ideal', 'Premium', 'Very Good']
color    7 niveles: ['D', 'E', 'F', 'G', 'H', 'I', 'J']
clarity  8 niveles: ['I1', 'IF', 'SI1', 'SI2', 'VS1', 'VS2', 'VVS1', 'VVS2']


Ese es el orden **alfabético**, y no tiene nada que ver con la calidad. En color,
`D` es el mejor y `J` el peor. Si dejamos que la máquina ordene, el orden sale
justo al revés.

In [39]:
from sklearn.preprocessing import OrdinalEncoder

# de peor a mejor; este orden lo pone quien sabe de diamantes, no el código
orden_cut     = ["Fair", "Good", "Very Good", "Premium", "Ideal"]
orden_color   = ["J", "I", "H", "G", "F", "E", "D"]
orden_clarity = ["I1", "SI2", "SI1", "VS2", "VS1", "VVS2", "VVS1", "IF"]

oe = OrdinalEncoder(categories=[orden_cut, orden_color, orden_clarity])
cod = oe.fit_transform(dia[["cut", "color", "clarity"]])

pd.DataFrame(cod, columns=["cut", "color", "clarity"]).head().astype(int)

,cut,color,clarity
0,4,5,1
1,3,5,2
2,1,5,4
3,3,1,3
4,1,0,1


In [40]:
tmp = dia.copy()
tmp[["cut_o", "color_o", "clarity_o"]] = cod
tmp[["carat", "cut_o", "color_o", "clarity_o", "price"]].corr()["price"].round(3)

carat        0.922
cut_o       -0.053
color_o     -0.173
clarity_o   -0.147
price        1.000
Name: price, dtype: float64

Curioso: la correlación de la calidad con el precio sale **negativa**. No es un
error de codificación — es que los diamantes grandes (que son los caros) suelen
tallarse con menos exigencia de calidad. Un buen recordatorio de que la
correlación simple engaña cuando hay una variable de confusión como `carat`.

In [41]:
print("one-hot de cut+color+clarity :",
      pd.get_dummies(dia[["cut", "color", "clarity"]]).shape[1], "columnas")
print("ordinal de cut+color+clarity :", 3, "columnas")

one-hot de cut+color+clarity : 20 columnas
ordinal de cut+color+clarity : 3 columnas


**Ordinal gana cuando** el orden es real y el efecto es más o menos monótono.
**One-hot gana cuando** no hay orden, o el efecto no es monótono. En la duda, y
con datos suficientes, one-hot es la apuesta segura: no impone ninguna estructura
que no esté en los datos.

---
## Chequeo del capítulo 5 · Con las categóricas codificadas

Ya podemos meter las 7 columnas de texto. One-hot a todas, sin más.

In [42]:
from sklearn.compose import ColumnTransformer

def rama_categorica(min_frec):
    return Pipeline([("imp", SimpleImputer(strategy="constant",
                                           fill_value="Desconocido")),
                     ("oh",  OneHotEncoder(handle_unknown="ignore",
                                           drop="first",
                                           min_frequency=min_frec))])

pre_todas = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median"))]), num_cols),
    ("cat", rama_categorica(1), cat_cols),
])

print("columnas tras codificar:",
      pre_todas.fit(X).transform(X).shape[1], f"  (para {len(X)} coches)")

chequeo("5 · + one-hot de todas",
        lambda m: Pipeline([("pre", pre_todas), ("reg", m)]), X, y)

columnas tras codificar: 62   (para 93 coches)


,capítulo,MSE lineal,R2 lineal,MSE red,R2 red
0,1a · Sin tocar nada,error,error,error,error
1,1b · Solo las numéricas,error,error,error,error
2,2 · dropna sobre las numéricas,46.77,0.295,68165.39,-1015.81
3,3a · Imputando fuera (con fuga),37.39,0.44,71652.15,-1132.738
4,3b · Imputando dentro (correcto),37.39,0.44,71652.37,-1132.739
5,4 · SimpleImputer(mediana),37.39,0.44,71652.37,-1132.739
6,5 · + one-hot de todas,91.1,-0.242,18748.45,-389.281


**El $R^2$ se volvió negativo:** de 0,440 a −0,242. Añadir información hizo el
modelo *peor que predecir siempre la media*. Y no es ruido, es aritmética: 62
columnas para 93 coches.

El culpable tiene nombre: `Manufacturer`, con sus 31 niveles, aporta 30 de esas
62 columnas. La red "mejora" de −1132 a −389, pero sigue siendo un disparate;
que baje no significa que sirva.

La codificación no estaba mal. Lo que falta es **controlar la cardinalidad**.

---
## 6 · Agrupación y discretización

In [43]:
(cars_m.groupby("Type")
       .agg(n=("Price", "size"),
            precio_medio=("Price", "mean"),
            cv_medio=("MPG.city", "mean"),
            hp_medio=("Horsepower", "mean"))
       .round(1)
       .sort_values("precio_medio", ascending=False))

,n,precio_medio,cv_medio,hp_medio
Type,,,,
Midsize,22,27.2,19.5,173.1
Large,11,24.3,18.4,179.5
Sporty,14,19.4,21.8,160.1
Van,9,19.1,17.0,149.4
Compact,16,18.2,22.7,131.0
Small,21,10.2,29.9,91.0


`Type` separa muy bien el precio: de 10,2 a 27,2. Es una variable que vale la
pena conservar.

In [44]:
gama = pd.cut(cars_m["Price"], bins=[0, 12, 20, 100],
              labels=["económico", "medio", "alto"])
gama.value_counts().sort_index()

Price
económico    22
medio        40
alto         31
Name: count, dtype: int64

- **`pd.cut`** — cortes que ustedes eligen. Úsenlo cuando los umbrales significan
  algo.
- **`pd.qcut`** — cortes por cuantiles: grupos del mismo tamaño.
- **`KBinsDiscretizer`** — la versión de sklearn, encajable en un `Pipeline`.

> ⚠️ Discretizar **tira información**: dos coches de 11.900 y 12.100 acaban en
> gamas distintas, y uno de 12.100 y otro de 19.900 acaban en la misma. Se
> justifica cuando la relación no es lineal y el modelo sí, o cuando hay que
> comunicar el resultado. No se justifica "para simplificar".

---
## Chequeo del capítulo 6 · Agrupando las categorías raras

Una categoría necesita **15 coches** para tener columna propia. Las demás se
juntan en un grupo.

In [45]:
pre_agrupadas = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median"))]), num_cols),
    ("cat", rama_categorica(15), cat_cols),      # <- el cambio
])

print("columnas:", pre_todas.fit(X).transform(X).shape[1], "->",
      pre_agrupadas.fit(X).transform(X).shape[1])

chequeo("6 · + agrupar categorías raras",
        lambda m: Pipeline([("pre", pre_agrupadas), ("reg", m)]), X, y)

columnas: 62 -> 26


,capítulo,MSE lineal,R2 lineal,MSE red,R2 red
0,1a · Sin tocar nada,error,error,error,error
1,1b · Solo las numéricas,error,error,error,error
2,2 · dropna sobre las numéricas,46.77,0.295,68165.39,-1015.81
3,3a · Imputando fuera (con fuga),37.39,0.44,71652.15,-1132.738
4,3b · Imputando dentro (correcto),37.39,0.44,71652.37,-1132.739
5,4 · SimpleImputer(mediana),37.39,0.44,71652.37,-1132.739
6,5 · + one-hot de todas,91.1,-0.242,18748.45,-389.281
7,6 · + agrupar categorías raras,33.3,0.455,26326.33,-377.586


**Recuperado, y mejor que antes.** El $R^2$ pasa de −0,242 a 0,455: no solo
deshacemos el destrozo del capítulo anterior, sino que superamos el 0,440 que
teníamos sin categóricas. Ahora sí aportan.

La lección no es "one-hot es peligroso". Es que **una técnica correcta, aplicada
sin mirar lo que produce, hace daño**. Bastó contar cuántos coches había por
marca.

Seis capítulos y la red sigue en −377.

---
## 7 · Escalado

Cualquier algoritmo basado en **distancias** (KNN, k-means, SVM) o en **descenso
de gradiente** verá que `Weight` domina, simplemente porque sus números son más
grandes. No porque importe más.

In [46]:
cols = ["Horsepower", "MPG.city", "Weight"]
X_train[cols].describe().loc[["mean", "std", "min", "max"]].round(2)

,Horsepower,MPG.city,Weight
mean,141.20,22.24,3093.11
std,48.03,5.70,592.10
min,55.00,15.00,1695.00
max,300.00,46.00,4105.00


### 7.1 Y se mide

In [47]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

sin = Pipeline([("imp", SimpleImputer()),
                ("knn", KNeighborsRegressor(5))]).fit(X_train[num_cols], y_train)

con = Pipeline([("imp", SimpleImputer()),
                ("esc", StandardScaler()),
                ("knn", KNeighborsRegressor(5))]).fit(X_train[num_cols], y_train)

print(f"KNN sin escalar : MAE {mean_absolute_error(y_test, sin.predict(X_test[num_cols])):.2f}")
print(f"KNN escalado    : MAE {mean_absolute_error(y_test, con.predict(X_test[num_cols])):.2f}")

KNN sin escalar : MAE 4.39
KNN escalado    : MAE 3.29


Un 10 % mejor sin tocar el modelo. Con variables de escalas más dispares la
diferencia es mucho mayor.

### 7.2 Los tres escaladores

| | Fórmula | Resultado |
|---|---|---|
| `StandardScaler` | $(x - \mu) / \sigma$ | media 0, desviación 1 |
| `MinMaxScaler` | $(x - x_{min}) / (x_{max} - x_{min})$ | todo entre 0 y 1 |
| `RobustScaler` | $(x - \text{mediana}) / (Q_3 - Q_1)$ | centro 0, robusto a atípicos |

In [48]:
from sklearn.preprocessing import MinMaxScaler, RobustScaler

std = StandardScaler().fit(X_train[cols])
mm  = MinMaxScaler().fit(X_train[cols])
rb  = RobustScaler().fit(X_train[cols])

for nombre, esc in [("StandardScaler", std), ("MinMaxScaler", mm), ("RobustScaler", rb)]:
    print(f"\n── {nombre} ──")
    print(pd.DataFrame(esc.transform(X_train[cols]), columns=cols)
            .describe().loc[["mean", "std", "min", "max"]].round(2))


── StandardScaler ──
      Horsepower  MPG.city  Weight
mean       -0.00      0.00    0.00
std         1.01      1.01    1.01
min        -1.81     -1.28   -2.38
max         3.33      4.20    1.72

── MinMaxScaler ──
      Horsepower  MPG.city  Weight
mean        0.35      0.23    0.58
std         0.20      0.18    0.25
min         0.00      0.00    0.00
max         1.00      1.00    1.00

── RobustScaler ──
      Horsepower  MPG.city  Weight
mean        0.02      0.25    0.03
std         0.75      1.14    0.68
min        -1.33     -1.20   -1.57
max         2.50      5.00    1.19


- **`StandardScaler`** no acota el rango: los atípicos siguen lejos (ese 4,20 es
  un coche de 46 mpg).
- **`MinMaxScaler`** garantiza el rango, pero **un solo atípico comprime a todos
  los demás**: miren la media de `MPG.city`, aplastada en 0,23.
- **`RobustScaler`** deja el centro en 0 y el grueso entre −1 y 1, **sin que los
  atípicos muevan la referencia**.

**Cuál usar:** `StandardScaler` como defecto razonable. `MinMaxScaler` cuando
hace falta un rango acotado y no hay atípicos. `RobustScaler` cuando hay atípicos
de verdad. **Ninguno** con árboles, bosques aleatorios o *gradient boosting*: les
da exactamente igual.

### 7.3 Lo que el escalador aprendió

In [49]:
print("media aprendida en train :", std.mean_.round(2))
print("desviación aprendida     :", std.scale_.round(2))

media aprendida en train : [ 141.2    22.24 3093.11]
desviación aprendida     : [ 47.7    5.66 588.08]


Esos seis números **son el modelo de preprocesamiento**. Hay que guardarlos y
llevárselos a producción junto con el modelo. Si se pierden, las predicciones
futuras no significan nada.

### 7.4 El test no queda entre 0 y 1, y está bien

In [50]:
pd.DataFrame(mm.transform(X_test[cols]), columns=cols).describe().loc[["min", "max"]].round(2)

,Horsepower,MPG.city,Weight
min,0.08,0.00,0.15
max,1.00,0.55,0.93


El máximo de `MPG.city` en test es 0,55 porque el coche más eficiente estaba en
entrenamiento. Si un dato de test superara el máximo de train, saldría **mayor
que 1**. No es un error: es la prueba de que no hubo fuga.

---
## Chequeo del capítulo 7 · Con escalado

Una línea más en la rama numérica: `StandardScaler()`.

In [51]:
pre_escalado = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("esc", StandardScaler())]), num_cols),
    ("cat", rama_categorica(15), cat_cols),
])

chequeo("7 · + escalado",
        lambda m: Pipeline([("pre", pre_escalado), ("reg", m)]), X, y)

,capítulo,MSE lineal,R2 lineal,MSE red,R2 red
0,1a · Sin tocar nada,error,error,error,error
1,1b · Solo las numéricas,error,error,error,error
2,2 · dropna sobre las numéricas,46.77,0.295,68165.39,-1015.81
3,3a · Imputando fuera (con fuga),37.39,0.44,71652.15,-1132.738
4,3b · Imputando dentro (correcto),37.39,0.44,71652.37,-1132.739
5,4 · SimpleImputer(mediana),37.39,0.44,71652.37,-1132.739
6,5 · + one-hot de todas,91.1,-0.242,18748.45,-389.281
7,6 · + agrupar categorías raras,33.3,0.455,26326.33,-377.586
8,7 · + escalado,33.3,0.455,35.23,0.449


### Esta es la celda de la sesión

La red pasa de $R^2 = -377{,}6$ a $R^2 = +0{,}449$. El MSE cae de **26.326 a
35**: setecientas veces menos error. La regresión lineal, en la misma línea, no
se mueve **ni una milésima**.

- **Por qué a la lineal le da igual.** Resuelve un sistema de ecuaciones. Si una
  columna vale mil veces más, su coeficiente vale mil veces menos y el resultado
  es idéntico.
- **Por qué a la red le cambia todo.** Entrena a base de pasos pequeños, y con
  las columnas en escalas incomparables la dirección de mejora es casi imposible
  de encontrar. Vamos a verlo por dentro.

In [52]:
# Abrimos las dos redes y comparamos qué hicieron por dentro
from sklearn.model_selection import KFold

Xn = X[num_cols].dropna(); yn = y.loc[Xn.index]
tr, te = next(iter(KFold(5).split(Xn)))

for etiqueta, pasos in [("SIN escalar", []), ("CON escalar", [("esc", StandardScaler())])]:
    m = Pipeline(pasos + [("red", MLPRegressor(hidden_layer_sizes=(32,),
                                               max_iter=2000,
                                               random_state=7))])
    m.fit(Xn.iloc[tr], yn.iloc[tr])
    r = m.named_steps["red"]
    p = m.predict(Xn.iloc[te])
    print(f"{etiqueta}")
    print(f"   iteraciones usadas : {r.n_iter_} de 2000")
    print(f"   pérdida final      : {r.loss_:.3g}")
    print(f"   peso mayor         : {np.abs(r.coefs_[0]).max():.3f}")
    print(f"   predice de {p.min():7.1f} a {p.max():7.1f}"
          f"   (los coches cuestan de {yn.iloc[te].min():.1f} a {yn.iloc[te].max():.1f})")

SIN escalar
   iteraciones usadas : 42 de 2000
   pérdida final      : 3.11e+04
   peso mayor         : 0.376
   predice de  -500.7 a    96.3   (los coches cuestan de 11.4 a 40.1)


CON escalar
   iteraciones usadas : 2000 de 2000
   pérdida final      : 2.99
   peso mayor         : 1.978
   predice de    11.3 a    40.9   (los coches cuestan de 11.4 a 40.1)


### No es que explote: es que se rinde

Lo primero que sorprende: **los pesos no se disparan**. El mayor vale 0,375, más
pequeño que el 1,978 de la versión escalada. La red no revienta.

Lo que pasa es que **se para sola en la iteración 40 de 2000**. La pérdida cae
rápido al principio —de 690.000 a 33.000— y ahí se estanca. `MLPRegressor` deja
de entrenar cuando la pérdida no mejora más de `tol=1e-4` durante 10 rondas
seguidas, y eso ocurre enseguida. Se detiene creyendo que ha terminado, con una
pérdida de 33.000 y prediciendo precios negativos.

La escalada, en cambio, agota las 2000 iteraciones: **todavía estaba mejorando**
cuando se le acabó el permiso.

In [53]:
# De ahí sale un R2 tan negativo: R2 = 1 - SS_res / SS_tot
m = MLPRegressor(hidden_layer_sizes=(32,), max_iter=2000,
                 random_state=7).fit(Xn.iloc[tr], yn.iloc[tr])
p, yv = m.predict(Xn.iloc[te]), yn.iloc[te]

ss_res = ((yv - p) ** 2).sum()
ss_tot = ((yv - yv.mean()) ** 2).sum()
print(f"SS_res (error del modelo)     {ss_res:12,.0f}")
print(f"SS_tot (varianza del precio)  {ss_tot:12,.0f}")
print(f"R2 = 1 - {ss_res/ss_tot:.1f} = {1 - ss_res/ss_tot:.1f}")

SS_res (error del modelo)        2,102,083
SS_tot (varianza del precio)         1,343
R2 = 1 - 1565.3 = -1564.3


El denominador es **pequeño**: los precios se mueven en un rango estrecho, así
que hay poca varianza que explicar. El numerador es enorme porque la red falla
por cientos de miles. Un cociente de mil y pico da un $R^2$ de mil y pico en
negativo.

Un $R^2$ muy negativo no significa "mil veces peor que la media" en ningún
sentido intuitivo: significa que el error es de **otro orden de magnitud** que la
variación que se pretendía explicar.

> No existe "el preprocesamiento correcto". Existe **el correcto para el modelo
> que van a usar**.

---
## 8 · Transformar variables sesgadas

In [54]:
print(f"asimetría de price      : {dia['price'].skew():.3f}")
print(f"asimetría de log(price) : {np.log1p(dia['price']).skew():.3f}")
print()
print(dia["price"].describe().round(1))

asimetría de price      : 1.618
asimetría de log(price) : 0.116

count    53940.0
mean      3932.8
std       3989.4
min        326.0
25%        950.0
50%       2401.0
75%       5324.2
max      18823.0
Name: price, dtype: float64


Media 3.933 y mediana 2.401: la media va muy por encima, señal clara de cola a la
derecha. El logaritmo la deja casi simétrica.

**Por qué molesta el sesgo.** La regresión lineal minimiza el **error al
cuadrado**. Con una cola larga, los pocos diamantes carísimos aportan errores
enormes al elevarlos al cuadrado, y el ajuste entero se va detrás de ellos. Al
trabajar en escala logarítmica, un error del 10 % pesa lo mismo en un diamante de
500 dólares que en uno de 15.000.

**Herramientas:**

- **`np.log1p`** — variables positivas con cola derecha. El `1p` es para que
  aguante los ceros.
- **`PowerTransformer`** — Yeo-Johnson busca el exponente que más acerca a la
  normal. Admite negativos.
- **`QuantileTransformer`** — fuerza la distribución que se le pida. Muy
  agresivo; puede destruir relaciones reales.

> ⚠️ Si transforman la **variable objetivo**, las predicciones salen en escala
> logarítmica: hay que deshacer con `np.expm1` antes de interpretar el error.

---
## Chequeo del capítulo 8 · Con el objetivo en logaritmo

In [55]:
from sklearn.compose import TransformedTargetRegressor

chequeo("8 · + log del objetivo",
        lambda m: TransformedTargetRegressor(
            regressor=Pipeline([("pre", pre_escalado), ("reg", m)]),
            func=np.log1p, inverse_func=np.expm1),
        X, y)

,capítulo,MSE lineal,R2 lineal,MSE red,R2 red
0,1a · Sin tocar nada,error,error,error,error
1,1b · Solo las numéricas,error,error,error,error
2,2 · dropna sobre las numéricas,46.77,0.295,68165.39,-1015.81
3,3a · Imputando fuera (con fuga),37.39,0.44,71652.15,-1132.738
4,3b · Imputando dentro (correcto),37.39,0.44,71652.37,-1132.739
5,4 · SimpleImputer(mediana),37.39,0.44,71652.37,-1132.739
6,5 · + one-hot de todas,91.1,-0.242,18748.45,-389.281
7,6 · + agrupar categorías raras,33.3,0.455,26326.33,-377.586
8,7 · + escalado,33.3,0.455,35.23,0.449
9,8 · + log del objetivo,29.59,0.581,334.16,-3.704


La lineal firma su mejor resultado de la sesión —$R^2 = 0{,}581$— y la red se
desploma de 0,449 a −3,7.

**El mismo aviso, al revés.** El logaritmo ayuda a la regresión lineal porque la
relación entre las características y el precio se vuelve más lineal en esa
escala. A la red no le hacía falta: ya podía curvarse sola, y lo único que le
dimos fue un objetivo más difícil de deshacer.

En el capítulo 7 el escalado salvó a la red y no le hizo nada a la lineal. Aquí
pasa exactamente lo contrario. **Cada paso hay que probarlo con el modelo que se
va a usar**, y quedarse con el que mejore.

---
# La foto completa

Ocho capítulos, dos modelos, una tabla.

In [56]:
pd.DataFrame(historial, columns=["capítulo", "MSE lineal", "R2 lineal",
                                 "MSE red", "R2 red"])

,capítulo,MSE lineal,R2 lineal,MSE red,R2 red
0,1a · Sin tocar nada,error,error,error,error
1,1b · Solo las numéricas,error,error,error,error
2,2 · dropna sobre las numéricas,46.77,0.295,68165.39,-1015.81
3,3a · Imputando fuera (con fuga),37.39,0.44,71652.15,-1132.738
4,3b · Imputando dentro (correcto),37.39,0.44,71652.37,-1132.739
5,4 · SimpleImputer(mediana),37.39,0.44,71652.37,-1132.739
6,5 · + one-hot de todas,91.1,-0.242,18748.45,-389.281
7,6 · + agrupar categorías raras,33.3,0.455,26326.33,-377.586
8,7 · + escalado,33.3,0.455,35.23,0.449
9,8 · + log del objetivo,29.59,0.581,334.16,-3.704


Mismos dos modelos las ocho veces. Todo lo que cambia sale del preprocesamiento.

### Los dos recorridos no se parecen en nada

- **La regresión lineal** arranca en cuanto hay números y va mejorando poco a
  poco: 0,295 → 0,440 → 0,455 → 0,581. Solo tropieza una vez, con las 62
  columnas del capítulo 5.
- **La red neuronal** es inservible durante *seis capítulos* —R² entre −377 y
  −1132— y de golpe útil al escalar. Después el logaritmo vuelve a hundirla.

Si alguien mira solo la columna de la lineal, concluye que el escalado no sirve
para nada. Si mira solo la de la red, concluye que sin escalar no se puede hacer
nada. Las dos lecturas son falsas.

### Cinco cosas que deja esta tabla

1. **Cada paso resuelve un problema distinto.** Imputar recupera filas.
   Codificar añade información. Agrupar controla las columnas. Escalar habilita
   modelos. Transformar linealiza.
2. **Un paso correcto puede empeorar.** El capítulo 5 hizo el $R^2$ negativo. La
   técnica estaba bien; faltaba mirar el resultado.
3. **Y otro puede no cambiar nada.** El capítulo 3 dio cifras idénticas con fuga
   y sin ella. Se sigue la regla igual.
4. **El mejor paso depende del modelo.** El escalado salvó a la red y no tocó a
   la lineal. El logaritmo, al revés.
5. **Sin medir, nada de esto se ve.** Ocho chequeos son ocho oportunidades de
   descubrir que el último cambio fue en la dirección equivocada.

### El error que esta tabla evita

Un estudiante que pruebe la red en el capítulo 6 y vea un $R^2$ de −377
concluirá que *"las redes no sirven para datos tabulares"* o que *"faltan
datos"*. Las dos conclusiones son falsas, y las dos son caras. Lo que hacía
falta era una línea: `StandardScaler()`.

Por eso el orden es **preparar los datos, medir, y solo entonces cambiar de
modelo**.

### Lo que no hemos hecho

Ninguno de los dos modelos está **afinado**. La red lleva 32 neuronas y 2000
iteraciones porque hacía falta fijar algo, no porque sea la mejor opción. Con los
datos ya preparados, un $R^2$ de 0,581 es el punto de partida, no el final:
elegir y ajustar el modelo es la sesión siguiente.

---
## Cierre

### Lista de comprobación

1. Mirar los datos: `shape`, `head`, `info`, `describe`.
2. Entender **por qué** falta lo que falta, antes de rellenarlo.
3. **Partir en entrenamiento y prueba.**
4. Imputar: mediana / moda / categoría nueva, con `fit` solo en entrenamiento.
5. Codificar: binaria -> 0-1; ordinal -> orden declarado; nominal -> one-hot.
6. Vigilar la cardinalidad y agrupar las categorías raras.
7. Escalar si el modelo lo necesita.
8. Transformar lo muy sesgado.
9. Encapsular todo en un `Pipeline`.
10. Comparar contra una línea base y validar de forma cruzada.

### Los tres errores que más van a ver

1. **`fit` sobre todo el dataset.** El más común y el más silencioso: no da
   error, solo resultados demasiado buenos que luego no se reproducen.
2. **Label encoding en nominales.** Le inventa un orden y unas distancias a lo
   que no las tiene.
3. **One-hot sin mirar la cardinalidad.** Como acabamos de ver, hunde el modelo
   sin avisar.

---

## Ejercicio propuesto

Repitan las tres etapas sobre **Ames Housing**: 1.460 casas, 80 columnas, vacíos
por todas partes y categóricas ordinales de verdad.

```python
from sklearn.datasets import fetch_openml
ames = fetch_openml(name="house_prices", as_frame=True).frame
```

Tres preguntas para guiarse:

1. ¿Qué columnas tienen tanto vacío que no vale la pena imputarlas?
2. ¿Cuáles de las categóricas son ordinales de verdad? (Pista: hay varias que
   van de `Po` a `Ex`.)
3. Con 1.460 filas en vez de 93, ¿sigue haciendo falta `min_frequency`?